# EndoScan ER — Phase 2b operator runbook (FOUR STOPS — *not* Run-all)

**You are the operator. You never edit Python.** You only **paste values** into
the clearly-marked CONFIG cells in Stop 2 and Stop 3. Run the cells of one stop,
then **STOP**: copy the printed outputs into Claude Chat and wait for the next
instruction. Do **not** run the next stop until Claude Chat tells you to.

- **Stop 1** — fetch the approved sources and print what they look like. (No
  staging, no training.)
- **Stop 2** — paste the column names Claude Chat gives you; build the dataset and
  run the **quality gate only** (training is intentionally impossible here).
- **Stop 3** — paste the metric floors + approval Claude Chat gives you; train.
- **Stop 4** — only after Claude Chat approves: push artifacts and print the exact
  commit commands for a separate review PR.

EndoScan is a pre-screening / prioritization tool — **no regulatory-grade claims**.

## STOP 1 — Fetch & inspect
Run the three cells below in order, then STOP.

In [ ]:
# 1a) Environment (NumPy<2 so cmapPy works) + repo.
!pip -q install 'numpy<2' cmapPy h5py pyarrow requests dvc 'scikit-learn>=1.8' pydantic pyyaml pandas
!git clone https://github.com/Rirkella/endoscan-platform.git
%cd endoscan-platform
!pip -q install -e packages/endoscan_core
import sys; sys.path.insert(0, 'pipelines/endpoints/ER/staging')

In [ ]:
# 1b) Mount Google Drive and configure the DVC LOCAL remote (no credentials in repo;
#     the path lives in git-ignored .dvc/config.local).
from google.colab import drive; drive.mount('/content/drive')
!dvc remote add --local er_gdrive /content/drive/MyDrive/endoscan-dvc || true
!dvc remote default --local er_gdrive
print('DVC local remote -> /content/drive/MyDrive/endoscan-dvc')

In [ ]:
# 1c) Fetch the approved sources and INSPECT them. Nothing is staged or trained.
from pathlib import Path
import gzip, shutil, pandas as pd, h5py
import fetch
from endoscan_core.datasets import load_sources

loc = {l.name: l.url for s in load_sources().sources for l in s.locators}
raw = Path('data/raw'); raw.mkdir(parents=True, exist_ok=True)

# Three LINCS metadata files (headers to confirm with Claude Chat). pert_info is
# included because it carries the perturbagen -> InChIKey link used to match LINCS
# signatures to CERAPP-labelled compounds.
small = {
    'sig_info':  loc['gse92742_sig_info'],
    'gene_info': loc['gse92742_gene_info'],
    'pert_info': loc['gse92742_pert_info'],
}
for name, url in small.items():
    fetch.fetch_url(url, raw / f'{name}.txt.gz')
    head = pd.read_csv(raw / f'{name}.txt.gz', sep='\t', nrows=5)
    n = sum(1 for _ in gzip.open(raw / f'{name}.txt.gz', 'rt')) - 1
    print(f'\n=== {name}: {n} rows ===\ncolumns: {list(head.columns)}'); display(head.head())

# CERAPP EXPERIMENTAL label file: download from the figshare/gaftp locator in
# sources.yaml and upload to data/raw/cerapp_experimental.csv, then re-run 1c.
print('\nCERAPP locator(s):', [k for k in loc if 'cerapp' in k])
cerapp_path = raw / 'cerapp_experimental.csv'
if cerapp_path.exists():
    c = pd.read_csv(cerapp_path, nrows=5)
    print(f'\n=== cerapp_experimental: {len(pd.read_csv(cerapp_path))} rows ===\ncolumns: {list(c.columns)}')
    display(c.head())
else:
    print('Upload the CERAPP EXPERIMENTAL set to', cerapp_path, 'then re-run 1c.')

# Level-5 MODZ gctx (large) — download last; read shape from HDF5 meta only.
fetch.fetch_url(loc['gse92742_level5_modz_gctx'], raw / 'level5_modz.gctx.gz')
with gzip.open(raw / 'level5_modz.gctx.gz', 'rb') as fi, open(raw / 'level5_modz.gctx', 'wb') as fo:
    shutil.copyfileobj(fi, fo)
with h5py.File(raw / 'level5_modz.gctx', 'r') as f:
    n_genes = f['/0/META/ROW/id'].shape[0]; n_sigs = f['/0/META/COL/id'].shape[0]
print(f'\n=== gctx shape: {n_genes} genes x {n_sigs} signatures ===')
print('Landmark count = number of gene_info rows with the landmark flag set (confirm\n'
      'the flag column with Claude Chat in Stop 2; expected 978).')

### ⛔ STOP — send Claude Chat:
the printed **headers + row counts** for all four files (`sig_info`, `gene_info`,
`pert_info`, `cerapp_experimental`) and the **gctx shape + landmark count**.
**Do not proceed** until Claude Chat replies with the confirmed column names.

## STOP 2 — Paste reviewed columns, stage, gate ONLY
Paste the values Claude Chat gave you into the CONFIG cell, run both cells, then
STOP. **Training cannot happen in this stop** (approval is hard-wired to False).

In [ ]:
# 2a) CONFIG — paste ONLY these values (column names Claude Chat confirmed). No logic.
SIG_ID_COL     = ''   # sig_info: signature id
SIG_PERT_COL   = ''   # sig_info: perturbagen/compound id
SIG_CELL_COL   = ''   # sig_info: cell line
SIG_DOSE_COL   = ''   # sig_info: dose
SIG_TIME_COL   = ''   # sig_info: time
GENE_LM_COL    = ''   # gene_info: landmark flag column
GENE_ID_COL    = ''   # gene_info: gene id
GENE_SYM_COL   = ''   # gene_info: gene symbol
PERT_ID_COL    = ''   # pert_info: perturbagen id
PERT_INCHI_COL = ''   # pert_info: InChIKey
CERAPP_ACTIVITY_COL = ''  # cerapp: experimental ER activity call
CERAPP_CASRN_COL    = ''  # cerapp: CASRN

In [ ]:
# 2b) Build staged files (tested helpers) and run the GATE ONLY (no training).
import importlib.util, pandas as pd
from pathlib import Path
import cerapp, pubchem, stage_er, fetch
from endoscan_core.datasets import load_sources

raw = Path('data/raw'); staged = Path('data/staged/er'); staged.mkdir(parents=True, exist_ok=True)

# Labels: CERAPP EXPERIMENTAL calls only (never consensus predictions).
cerapp_rows = pd.read_csv(raw / 'cerapp_experimental.csv').to_dict('records')
labels = cerapp.parse_cerapp_experimental(cerapp_rows, activity_col=CERAPP_ACTIVITY_COL,
                                          casrn_col=CERAPP_CASRN_COL)
pd.DataFrame(labels).to_csv(staged / 'cerapp.csv', index=False)

# CASRN -> InChIKey for the labelled compounds (PubChem REST), normalized + saved.
casrns = sorted({r['casrn'] for r in labels})
mapping_rows = pubchem.normalize_mapping(fetch.pubchem_mapping_for_casrns(casrns))
pd.DataFrame(mapping_rows).to_csv(staged / 'pubchem.csv', index=False)
casrn_to_inchi = {m['input_id']: m['inchikey'] for m in mapping_rows}
labelled_inchikeys = {casrn_to_inchi[r['casrn']] for r in labels if r['casrn'] in casrn_to_inchi}

# Perturbagen -> InChIKey from pert_info; assemble sig_meta (MCF7/A549, labelled only).
pert_info = pd.read_csv(raw / 'pert_info.txt.gz', sep='\t')
pert_to_inchi = dict(zip(pert_info[PERT_ID_COL].astype(str), pert_info[PERT_INCHI_COL].astype(str)))
sig_info = pd.read_csv(raw / 'sig_info.txt.gz', sep='\t')
sig_meta = stage_er.assemble_sig_meta(sig_info, pert_to_inchi, labelled_inchikeys,
    sig_id_col=SIG_ID_COL, pert_id_col=SIG_PERT_COL, cell_id_col=SIG_CELL_COL,
    dose_col=SIG_DOSE_COL, time_col=SIG_TIME_COL)

# Landmark genes + slice the gctx ONCE -> early-fused lincs.parquet.
gene_info = pd.read_csv(raw / 'gene_info.txt.gz', sep='\t')
gene_ids, gene_syms = stage_er.select_landmark_genes(gene_info, landmark_flag_col=GENE_LM_COL,
    gene_id_col=GENE_ID_COL, gene_symbol_col=GENE_SYM_COL)
stage_er.build_lincs_parquet(sig_meta, raw / 'level5_modz.gctx', gene_ids, gene_syms,
                             staged / 'lincs.parquet')

# GATE ONLY — approval hard-wired False here, so run_pipeline cannot train.
spec = importlib.util.spec_from_file_location('er_run', 'pipelines/endpoints/ER/run.py')
er_run = importlib.util.module_from_spec(spec); sys.modules['er_run'] = er_run; spec.loader.exec_module(er_run)
cfg = er_run.PipelineConfig.model_validate({
    'endpoint_id': 'ER', 'biological_target': 'Estrogen Receptor', 'version': '0.1.0',
    'data': {'target': 'ER', 'adapter': 'staged', 'staged_dir': 'data/staged/er', 'n_groups': 10, 'seed': 0},
    'gate': {'thresholds_path': 'registry/data/quality_gates.yaml'},
    'approval': {'approved': False},  # HARD BOUNDARY for Stop 2
    'evaluation': {'mode': 'nested', 'outer_splits': 5, 'inner_splits': 3},
})
res = er_run.run_pipeline(cfg, allow_list=load_sources(), data_dir=staged,
    thresholds=er_run.load_thresholds(Path('registry/data/quality_gates.yaml')), output_root=Path('.'))
assert res.trained is False  # structurally guaranteed in Stop 2
pos = sum(1 for r in labels if r['consensus_call'] == 'active')
print('gate:', res.gate_summary, '| overlap compounds:', res.n_overlap,
      '| labelled prevalence:', round(pos / max(len(labels), 1), 3))
print('\n=== dataset_card.md ===\n' + Path(res.dataset_card_path).read_text())

### ⛔ STOP — send Claude Chat:
the printed **`dataset_card.md`**, the **gate summary**, the **prevalence**, and the
**compound counts**. Training is intentionally unreachable here. Wait for Claude
Chat to provide the metric floors before Stop 3.

## STOP 3 — Paste floors + approval, train
Paste the floors Claude Chat gave you, run both cells, then STOP.

In [ ]:
# 3a) CONFIG — paste ONLY these values (validated_mvp floors Claude Chat provided).
FLOOR_AUROC             = 0.0   # <- paste
FLOOR_AUPRC             = 0.0   # <- paste (sized to prevalence)
FLOOR_BALANCED_ACCURACY = 0.0   # <- paste
CEIL_BRIER              = 1.0   # <- paste
APPROVED                = True  # Claude Chat authorizes training for this stop

In [ ]:
# 3b) Train: re-gates, trains (nested honest estimate), writes artifacts + proposed entry.
from pathlib import Path
from endoscan_core.datasets import load_sources
cfg = er_run.PipelineConfig.model_validate({
    'endpoint_id': 'ER', 'biological_target': 'Estrogen Receptor', 'version': '0.1.0',
    'data': {'target': 'ER', 'adapter': 'staged', 'staged_dir': 'data/staged/er', 'n_groups': 10, 'seed': 0},
    'gate': {'thresholds_path': 'registry/data/quality_gates.yaml'},
    'approval': {'approved': APPROVED, 'approved_by': 'operator+claude-chat'},
    'evaluation': {'mode': 'nested', 'outer_splits': 5, 'inner_splits': 3},
    'training': {'validated_mvp_floors': {'auroc': FLOOR_AUROC, 'auprc': FLOOR_AUPRC,
                                          'balanced_accuracy': FLOOR_BALANCED_ACCURACY},
                 'validated_mvp_ceilings': {'brier_score': CEIL_BRIER}},
})
res = er_run.run_pipeline(cfg, allow_list=load_sources(), data_dir=Path('data/staged/er'),
    thresholds=er_run.load_thresholds(Path('registry/data/quality_gates.yaml')), output_root=Path('.'))
print('trained:', res.trained, '| status:', res.status, '| model:', res.selected_model)
print('\n=== metrics.json ===\n' + Path('models/ER/metrics.json').read_text())
print('\n=== model_selection.md ===\n' + Path('models/ER/model_selection.md').read_text())

### ⛔ STOP — send Claude Chat:
`models/ER/metrics.json`, `model_selection.json`, `model_selection.md`,
`model_card.md`, `dataset_card.md`, `feature_schema.json`, and the proposed
`registry/models/endpoints.json` entry. **Do NOT push or commit yet.**

## STOP 4 — Only after Claude Chat approval: push + print commit commands

In [ ]:
# 4) DVC-push the binaries to the Drive remote, then PRINT the git commands.
#    (This cell pushes data; it does NOT commit or open a PR.)
!dvc add data/staged/er models/ER/model.pkl && dvc push
print('\nReviewed by Claude Chat? Then commit ONLY the text artifacts + .dvc pointers\n'
      'in a SEPARATE branch/PR (open it for Claude Chat final diff review before merge):\n')
print('git checkout -b er-real-endpoint-phase2b')
print('git add data/staged/er.dvc models/ER/model.pkl.dvc \\')
print('        models/ER/metrics.json models/ER/feature_schema.json \\')
print('        models/ER/model_card.md models/ER/dataset_card.md \\')
print('        models/ER/model_selection.json models/ER/model_selection.md \\')
print('        registry/models/endpoints.json')
print('git commit -m "ER real endpoint (Phase 2b): registered <status> with DVC artifacts"')
print('git push -u origin er-real-endpoint-phase2b   # then open a PR for review')